# 노지 작물 해충 진단 (서브셋) - Qwen3.5-9B LoRA 파인튜닝

**대상**: 썩덩나무노린재 + 정상 (2클래스)  
**데이터셋**: `data/` (로컬)  
**모델**: Qwen3.5-9B (bf16 LoRA)  
**환경**: 32GB+ VRAM (A5000/A6000)

---

In [ ]:
import os
import requests

DISCORD_BOT = {
    "username": "RunPod",
    "avatar_url": "https://i.imgur.com/0HOIh4r.png",
}
DISCORD_COLOR = 12648430
DISCORD_THUMBNAIL = "https://i.imgur.com/3ClKkzk.jpeg"


def notify_discord(message):
    """DISCORD_WEBHOOK_URL이 설정되어 있으면 마크다운 메시지 전송"""
    url = os.environ.get("DISCORD_WEBHOOK_URL")
    if not url:
        return
    try:
        requests.post(url, json={"content": message}, timeout=10)
    except Exception as e:
        print(f"Discord 알림 실패: {e}")


def notify_discord_json(payload):
    """DISCORD_WEBHOOK_URL이 설정되어 있으면 JSON payload를 그대로 전송 (Embed 등)"""
    url = os.environ.get("DISCORD_WEBHOOK_URL")
    if not url:
        return
    try:
        requests.post(url, json=payload, timeout=10)
    except Exception as e:
        print(f"Discord 알림 실패: {e}")


def discord_embed(description, thumbnail=False):
    """Embed payload 생성 헬퍼"""
    embed = {"description": description, "color": DISCORD_COLOR}
    if thumbnail:
        embed["thumbnail"] = {"url": DISCORD_THUMBNAIL}
    return {**DISCORD_BOT, "embeds": [embed]}


print("Discord Webhook:", "활성화" if os.environ.get("DISCORD_WEBHOOK_URL") else "비활성화 (DISCORD_WEBHOOK_URL 미설정)")

## 하이퍼파라미터 설정

아래 셀의 값만 바꾸면 전체 파이프라인에 반영됩니다.

In [ ]:
# ═══════════════════════════════════════
# 하이퍼파라미터 (이 셀만 수정하세요)
# ═══════════════════════════════════════

BATCH_SIZE = 6              # per_device_train_batch_size
GRAD_ACCUM = 2              # gradient_accumulation_steps
                            # → Total Batch Size = BATCH_SIZE × GRAD_ACCUM × GPU 수

LORA_R = 16                 # LoRA rank (클수록 표현력 ↑, VRAM ↑)
LORA_ALPHA = 16             # LoRA alpha (보통 r과 같거나 2배)

LEARNING_RATE = 2e-4        # 학습률
NUM_EPOCHS = 3              # 학습 에폭 수
WARMUP_STEPS = 50           # 워밍업 스텝

# 고유 run name 자동 생성 (파라미터 조합)
RUN_NAME = f"r{LORA_R}_a{LORA_ALPHA}_lr{LEARNING_RATE}_bs{BATCH_SIZE}x{GRAD_ACCUM}_ep{NUM_EPOCHS}"
OUTPUT_DIR = f"pest-detector-{RUN_NAME}"
LORA_DIR = f"pest-lora-{RUN_NAME}"

print(f"Total Batch Size: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}, Warmup: {WARMUP_STEPS}")
print(f"RUN_NAME:   {RUN_NAME}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"LORA_DIR:   {LORA_DIR}")

## 1. 패키지 설치

In [ ]:
notify_discord_json(discord_embed("📦 [1/10] 패키지 설치를 시작합니다.", thumbnail=True))
try:
    !pip install --upgrade pip
    !pip install --upgrade typing_extensions
    !pip install unsloth
    !pip install "transformers>=5.2"
    !pip install trl==0.22.2 datasets Pillow accelerate scikit-learn huggingface_hub wandb requests

    from unsloth import FastVisionModel
    print("Unsloth OK")
    notify_discord_json(discord_embed("✅ [1/10] 패키지 설치가 완료되었습니다."))
except Exception as e:
    notify_discord_json(discord_embed(f"❌ [1/10] 패키지 설치 중 에러 발생: {e}"))
    raise

## 2. 데이터셋 경로 설정

In [ ]:
notify_discord_json(discord_embed("📂 [2/10] 데이터셋 경로를 확인합니다."))
try:
    import os

    DATA_DIR = "/workspace/data"

    assert os.path.exists(os.path.join(DATA_DIR, "train.jsonl")), \
        f"데이터셋이 없습니다: {DATA_DIR}/train.jsonl"
    print(f"데이터셋 경로: {DATA_DIR}")
    notify_discord_json(discord_embed("✅ [2/10] 데이터셋 경로 확인 완료. (train.jsonl, val.jsonl)"))
except Exception as e:
    notify_discord_json(discord_embed(f"❌ [2/10] 데이터셋 경로 확인 실패: {e}"))
    raise

## 3. 이미지 전처리 (크롭 → 디스크 저장)

해충 이미지는 바운딩 박스 기반 크롭 (tight 50% / context 50%)  
정상 이미지는 원본 유지  
결과를 디스크에 저장하고 새로운 JSONL 생성 → **RAM을 거의 안 씀**

In [ ]:
notify_discord_json(discord_embed("🖼️ [3/10] 이미지 전처리를 시작합니다. (크롭 → 디스크 저장)"))
try:
    import json
    import os
    import random
    from PIL import Image

    Image.MAX_IMAGE_PIXELS = None

    PROMPTS = [
        "이 사진에 있는 해충의 이름을 알려주세요.",
        "이 벌레는 무엇인가요?",
        "사진 속 해충을 식별해주세요.",
        "이 작물에 있는 해충의 종류가 무엇인가요?",
        "이 사진에서 어떤 해충이 보이나요?",
    ]

    SYSTEM_MSG = (
        "당신은 작물 해충 식별 전문가입니다. "
        '사진을 보고 해충의 이름만 한국어로 답하세요. '
        '해충이 없으면 "정상"이라고만 답하세요. '
        "부가 설명 없이 이름만 출력하세요."
    )

    BBOX_GROW_STAGE = 33


    def crop_to_bbox(img, bbox, padding_ratio=0.0):
        xtl, ytl = bbox["xtl"], bbox["ytl"]
        xbr, ybr = bbox["xbr"], bbox["ybr"]
        bw, bh = xbr - xtl, ybr - ytl
        pad_x, pad_y = int(bw * padding_ratio), int(bh * padding_ratio)
        x1 = max(0, xtl - pad_x)
        y1 = max(0, ytl - pad_y)
        x2 = min(img.width, xbr + pad_x)
        y2 = min(img.height, ybr + pad_y)
        return img.crop((x1, y1, x2, y2))


    def find_label_json(split, class_name, img_filename):
        json_path = os.path.join(DATA_DIR, split, class_name, img_filename + ".json")
        if not os.path.exists(json_path):
            return None
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        for obj in data["annotations"]["object"]:
            if obj["grow"] == BBOX_GROW_STAGE and obj.get("points"):
                return obj["points"][0]
        return None


    def preprocess_split(split="train"):
        jsonl_path = os.path.join(DATA_DIR, f"{split}.jsonl")
        out_dir = os.path.join(DATA_DIR, f"{split}_cropped")
        out_jsonl = os.path.join(DATA_DIR, f"{split}_cropped.jsonl")

        if os.path.exists(out_jsonl):
            with open(out_jsonl, "r") as f:
                count = sum(1 for _ in f)
            print(f"  [{split}] 이미 전처리 완료: {count}건")
            return count

        with open(jsonl_path, "r", encoding="utf-8") as f:
            total = sum(1 for _ in f)

        out_file = open(out_jsonl, "w", encoding="utf-8")
        count = 0

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if (i + 1) % 500 == 0 or (i + 1) == total:
                    print(f"\r  [{split}] {i + 1}/{total} ({(i + 1) * 100 // total}%)", end="", flush=True)

                record = json.loads(line)
                messages = record["messages"]
                label = messages[-1]["content"][0]["text"]

                img_rel_path = None
                for msg in messages:
                    for content in msg["content"]:
                        if content["type"] == "image" and "image" in content:
                            img_rel_path = content["image"].replace("\\", "/")
                            break

                if img_rel_path is None:
                    continue

                parts = img_rel_path.split("/")
                class_name = parts[1]
                img_filename = parts[2]

                class_dir = os.path.join(out_dir, class_name)
                os.makedirs(class_dir, exist_ok=True)

                base_name = os.path.splitext(img_filename)[0]
                out_filename = f"{base_name}.jpg"
                out_path = os.path.join(class_dir, out_filename)
                out_rel_path = f"{split}_cropped/{class_name}/{out_filename}"

                img_path = os.path.join(DATA_DIR, img_rel_path)
                img = Image.open(img_path).convert("RGB")

                if label == "정상":
                    result = img
                else:
                    bbox = find_label_json(split, class_name, img_filename)
                    if bbox:
                        r = random.random()
                        if r < 0.5:
                            result = img
                        elif r < 0.75:
                            result = crop_to_bbox(img, bbox, padding_ratio=0.0)
                        else:
                            result = crop_to_bbox(img, bbox, padding_ratio=0.5)
                    else:
                        result = img

                result.save(out_path, "JPEG", quality=95)
                if result is not img:
                    result.close()
                img.close()

                new_record = {
                    "messages": [
                        {"role": "system", "content": [
                            {"type": "text", "text": SYSTEM_MSG}
                        ]},
                        {"role": "user", "content": [
                            {"type": "image", "image": out_rel_path},
                            {"type": "text", "text": random.choice(PROMPTS)},
                        ]},
                        {"role": "assistant", "content": [
                            {"type": "text", "text": label}
                        ]},
                    ]
                }
                out_file.write(json.dumps(new_record, ensure_ascii=False) + "\n")
                count += 1

        out_file.close()
        print(f"\n  [{split}] 완료: {count}건 → {out_dir}")
        return count


    random.seed(42)
    num_train = preprocess_split("train")
    num_val = preprocess_split("val")
    notify_discord_json(discord_embed(f"✅ [3/10] 전처리 완료! (train {num_train}건, val {num_val}건)"))
except Exception as e:
    notify_discord_json(discord_embed(f"❌ [3/10] 이미지 전처리 중 에러 발생: {e}"))
    raise

## 4. 데이터 로딩 (경로 기반 — RAM 절약)

In [ ]:
notify_discord_json(discord_embed("📊 [4/10] 데이터를 로딩합니다."))
try:
    import json
    import os
    import random
    from PIL import Image

    def load_dataset_from_cropped_jsonl(split="train"):
        jsonl_path = os.path.join(DATA_DIR, f"{split}_cropped.jsonl")
        dataset = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                for msg in record["messages"]:
                    for content in msg["content"]:
                        if content["type"] == "image" and "image" in content:
                            content["image"] = os.path.join(DATA_DIR, content["image"])
                dataset.append(record)
        random.shuffle(dataset)
        return dataset

    random.seed(42)
    train_dataset = load_dataset_from_cropped_jsonl("train")
    val_dataset = load_dataset_from_cropped_jsonl("val")
    print(f"Train: {len(train_dataset)}건, Val: {len(val_dataset)}건")
    notify_discord_json(discord_embed(f"✅ [4/10] 데이터 로딩 완료! (Train {len(train_dataset)}건, Val {len(val_dataset)}건)"))
except Exception as e:
    notify_discord_json(discord_embed(f"❌ [4/10] 데이터 로딩 중 에러 발생: {e}"))
    raise

## 5. 모델 로딩

In [ ]:
notify_discord_json(discord_embed("🤖 [5/10] Qwen3.5-9B 모델을 로딩합니다."))
try:
    import os
    os.environ["HF_HOME"] = "/workspace/hf_cache"
    os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"

    import torch
    from unsloth import FastVisionModel

    model, tokenizer = FastVisionModel.from_pretrained(
        "unsloth/Qwen3.5-9B",
        load_in_4bit=False,
        use_gradient_checkpointing="unsloth",
    )
    notify_discord_json(discord_embed("✅ [5/10] 모델 로딩 완료!"))
except Exception as e:
    notify_discord_json(discord_embed(f"❌ [5/10] 모델 로딩 중 에러 발생: {e}"))
    raise

## 6. LoRA 설정

In [ ]:
notify_discord_json(discord_embed(f"⚙️ [6/10] LoRA 어댑터를 설정합니다. (r={LORA_R}, alpha={LORA_ALPHA})"))
try:
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    notify_discord_json(discord_embed("✅ [6/10] LoRA 설정 완료!"))
except Exception as e:
    notify_discord_json(discord_embed(f"❌ [6/10] LoRA 설정 중 에러 발생: {e}"))
    raise

## 7. 학습

In [ ]:
notify_discord_json(discord_embed(f"@everyone\n🚀 [7/10] 학습을 시작합니다! ({NUM_EPOCHS} epochs, batch={BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM})"))
try:
    from unsloth.trainer import UnslothVisionDataCollator
    from trl import SFTTrainer, SFTConfig

    FastVisionModel.for_training(model)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=SFTConfig(
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            warmup_steps=WARMUP_STEPS,
            num_train_epochs=NUM_EPOCHS,
            learning_rate=LEARNING_RATE,
            bf16=True,
            logging_steps=10,
            save_strategy="epoch",
            eval_strategy="epoch",
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=42,
            output_dir=OUTPUT_DIR,
            report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
            run_name=RUN_NAME,
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_seq_length=2048,
            dataloader_num_workers=8,
            dataloader_pin_memory=True,
        ),
    )

    trainer.train(resume_from_checkpoint=True)
    notify_discord_json(discord_embed("@everyone\n✅ [7/10] 학습 완료!"))
except Exception as e:
    notify_discord_json(discord_embed(f"@everyone\n❌ [7/10] 학습 중 에러 발생: {e}"))
    raise

## 8. 모델 저장

In [ ]:
notify_discord_json(discord_embed("💾 [8/10] LoRA 어댑터를 저장합니다."))
try:
    model.save_pretrained(LORA_DIR)
    tokenizer.save_pretrained(LORA_DIR)
    print(f"LoRA 어댑터 저장 완료: {LORA_DIR}")
    notify_discord_json(discord_embed(f"✅ [8/10] 모델 저장 완료! ({LORA_DIR}/)"))
except Exception as e:
    notify_discord_json(discord_embed(f"❌ [8/10] 모델 저장 중 에러 발생: {e}"))
    raise

## 9. 평가 (6개 메트릭)

In [ ]:
notify_discord_json(discord_embed("@everyone\n📈 [9/10] test 데이터셋으로 평가를 시작합니다."))
try:
    import time
    import json
    from datetime import datetime
    from PIL import Image
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        confusion_matrix,
    )

    CLASS_NAMES = ["썩덩나무노린재", "정상"]

    # test.jsonl 로딩
    test_jsonl = os.path.join(DATA_DIR, "test.jsonl")
    samples = []
    with open(test_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            messages = record["messages"]
            label = messages[-1]["content"][0]["text"]
            img_rel_path = None
            for msg in messages:
                for content in msg["content"]:
                    if content["type"] == "image" and "image" in content:
                        img_rel_path = content["image"].replace("\\", "/")
                        break
            if img_rel_path:
                samples.append((os.path.join(DATA_DIR, img_rel_path), label))

    print(f"테스트 샘플: {len(samples)}건\n")

    # 추론
    y_true = []
    y_pred = []
    inference_times = []

    for i, (img_path, label) in enumerate(samples):
        t_start = time.time()

        image = Image.open(img_path).convert("RGB")
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_MSG}]},
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": "이 사진에 있는 해충의 이름을 알려주세요."},
            ]},
        ]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
        output = model.generate(**inputs, max_new_tokens=20, use_cache=True, temperature=0.1)
        generated_ids = output[0][inputs["input_ids"].shape[-1]:]
        pred = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        image.close()

        t_elapsed = time.time() - t_start
        inference_times.append(t_elapsed)

        matched = pred
        if pred not in CLASS_NAMES:
            for cls in CLASS_NAMES:
                if cls in pred:
                    matched = cls
                    break

        y_true.append(label)
        y_pred.append(matched)

        status = "O" if label == matched else "X"
        print(f"  [{i+1}/{len(samples)}] {status}  정답: {label:10s}  예측: {matched:10s}  ({t_elapsed:.2f}s)")

    # ═══════ 6개 평가 메트릭 ═══════
    print("\n" + "=" * 60)
    print("평가 결과 (6개 메트릭)")
    print("=" * 60)

    cm = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES)
    print("\n[1] Confusion Matrix:")
    print(f"{'':15s} {'예측:썩덩나무노린재':>15s} {'예측:정상':>10s}")
    for idx, cls in enumerate(CLASS_NAMES):
        print(f"{'실제:'+cls:15s} {cm[idx][0]:15d} {cm[idx][1]:10d}")

    acc = accuracy_score(y_true, y_pred)
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    print(f"\n[2] Accuracy: {acc:.4f} ({correct}/{len(y_true)})")

    prec_per_class = precision_score(y_true, y_pred, labels=CLASS_NAMES, average=None, zero_division=0)
    prec_macro = precision_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0)
    print(f"\n[3] Precision:")
    for cls, p in zip(CLASS_NAMES, prec_per_class):
        print(f"    {cls}: {p:.4f}")
    print(f"    Macro: {prec_macro:.4f}")

    rec_per_class = recall_score(y_true, y_pred, labels=CLASS_NAMES, average=None, zero_division=0)
    rec_macro = recall_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0)
    print(f"\n[4] Recall:")
    for cls, r in zip(CLASS_NAMES, rec_per_class):
        print(f"    {cls}: {r:.4f}")
    print(f"    Macro: {rec_macro:.4f}")

    f1_per_class = f1_score(y_true, y_pred, labels=CLASS_NAMES, average=None, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0)
    print(f"\n[5] Macro F1 Score:")
    for cls, f in zip(CLASS_NAMES, f1_per_class):
        print(f"    {cls}: {f:.4f}")
    print(f"    Macro: {f1_macro:.4f}")

    avg_time = sum(inference_times) / len(inference_times)
    total_time = sum(inference_times)
    print(f"\n[6] 추론 속도:")
    print(f"    총 소요 시간: {total_time:.1f}s")
    print(f"    이미지당 평균: {avg_time:.2f}s")
    print(f"    처리량: {len(samples)/total_time:.1f} img/s")

    wrong = [(t, p, s[0]) for (s, t, p) in zip(samples, y_true, y_pred) if t != p]
    if wrong:
        print(f"\n오답 {len(wrong)}건:")
        for t, p, path in wrong:
            print(f"  정답: {t:10s}  예측: {p:10s}  {os.path.basename(path)}")

    print("=" * 60)

    # 평가 결과를 JSON으로 저장 (Hub 업로드 시 모델과 함께 올라감)
    eval_results = {
        "timestamp": datetime.now().isoformat(),
        "test_samples": len(samples),
        "confusion_matrix": cm.tolist(),
        "accuracy": round(acc, 4),
        "precision": {cls: round(float(p), 4) for cls, p in zip(CLASS_NAMES, prec_per_class)},
        "precision_macro": round(float(prec_macro), 4),
        "recall": {cls: round(float(r), 4) for cls, r in zip(CLASS_NAMES, rec_per_class)},
        "recall_macro": round(float(rec_macro), 4),
        "f1": {cls: round(float(f), 4) for cls, f in zip(CLASS_NAMES, f1_per_class)},
        "f1_macro": round(float(f1_macro), 4),
        "inference_speed": {
            "avg_seconds_per_image": round(avg_time, 2),
            "images_per_second": round(len(samples) / total_time, 1),
            "total_seconds": round(total_time, 1),
        },
        "wrong_count": len(wrong),
    }
    eval_path = os.path.join(LORA_DIR, "evaluation_results.json")
    with open(eval_path, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, ensure_ascii=False, indent=2)
    print(f"\n평가 결과 저장: {eval_path}")

    notify_discord_json(discord_embed(
        f"@everyone\n✅ [9/10] 평가 완료!\n\n"
        f"Accuracy: {acc:.4f} ({correct}/{len(y_true)})\n"
        f"Precision (Macro): {prec_macro:.4f}\n"
        f"Recall (Macro): {rec_macro:.4f}\n"
        f"Macro F1: {f1_macro:.4f}\n"
        f"추론 속도: {avg_time:.2f}s/img ({len(samples)/total_time:.1f} img/s)\n"
        f"오답: {len(wrong)}건"
    ))
except Exception as e:
    notify_discord_json(discord_embed(f"@everyone\n❌ [9/10] 평가 중 에러 발생: {e}"))
    raise

## 10. HuggingFace Hub에 업로드

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    HUB_REPO = f"pest-{RUN_NAME}"
    notify_discord_json(discord_embed(f"☁️ [10/10] HuggingFace Hub에 업로드합니다. ({HUB_REPO})"))
    try:
        from huggingface_hub import HfApi, create_repo

        repo_url = create_repo(HUB_REPO, token=HF_TOKEN, exist_ok=True, private=False)
        api = HfApi(token=HF_TOKEN)
        # LORA_DIR 전체 업로드 (LoRA 어댑터 + tokenizer + evaluation_results.json)
        api.upload_folder(
            folder_path=LORA_DIR,
            repo_id=repo_url.repo_id,
            commit_message=f"Upload {RUN_NAME}",
        )
        uploaded_files = os.listdir(LORA_DIR)
        has_eval = "evaluation_results.json" in uploaded_files
        print(f"업로드 완료! 파일 {len(uploaded_files)}개 (evaluation_results.json {'포함' if has_eval else '없음'})")
        notify_discord_json(discord_embed(
            f"✅ [10/10] 업로드 완료! ({HUB_REPO}) 🎉
"
            f"파일 {len(uploaded_files)}개 (evaluation_results.json {'포함' if has_eval else '없음'})"
        ))
    except Exception as e:
        notify_discord_json(discord_embed(f"❌ [10/10] HuggingFace Hub 업로드 중 에러 발생: {e}"))
        raise
else:
    print("HF_TOKEN 미설정 — 업로드를 건너뜁니다.")
    notify_discord_json(discord_embed("✅ [10/10] Hub 업로드 건너뜀 (HF_TOKEN 미설정). 파이프라인 완료! 🎉"))


In [ ]:
!rm -rf /root/.cache
!rm -rf /tmp/*
!df -h /
